# 07 Evidence check with System-1 decision models (Kaggle T4)

Light's top-20 pages per query are read one at a time by a calibrated yes/no model (Kev-0.8B and Laya-multilingual, both run with their own released code).
Each page gets P(the page contains the information needed to answer the question). A query-only question is asked too, as a control.
Routing rules and scoring run locally afterwards; this notebook only produces probabilities.

Input: attach `system1_input.jsonl` as a dataset (214 rows: query, group, light top-20 page ids and text). Settings: GPU T4, Internet ON.
Output: `system1_out_07.zip` with `kev_scores.jsonl`, `laya_scores.jsonl` and logs.

## 1. Settings

In [ ]:
INPUT_NAME = "system1_input.jsonl"
OUT = "/kaggle/working/system1_out"
ZIP = "system1_out_07.zip"

KEV_REPO = "https://github.com/jaredpalmer/kev"
KEV_COMMIT = "5e42a7a03f28134853dd3ff77461457e921e5ec1"
KEV_DIR = "/kaggle/working/kev"
KEV_MODEL = "jaredpalmer/kev-0.8b@v1.0"
KEV_DTYPE = "fp32"
KEV_PORT = 8009
KEV_WORKERS = 8

LAYA_SPEC = "git+https://github.com/NandhaKishorM/laya.git@1adc59f7e371deb601fcfa18a14e25db238addcc"
LAYA_MODEL = "multilingual"
LAYA_BATCH = 32

PAGE_QUESTION = "Does the page contain the information needed to answer the question?"
QUERY_QUESTION = "Does answering this question require reading figures, tables, images or handwriting?"

RUN_KEV = True
RUN_LAYA = True

## 2. Input

In [ ]:
import glob
import json
import os
import subprocess
import time

found = sorted(glob.glob(f"/kaggle/input/**/{INPUT_NAME}", recursive=True))
if not found:
    raise SystemExit(f"attach a dataset containing {INPUT_NAME}")
rows = [json.loads(line) for line in open(found[0])]
print(found[0], len(rows), "queries", sum(len(r["pages"]) for r in rows), "pages")
os.makedirs(f"{OUT}/logs", exist_ok=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)


def page_state(row, page):
    return {"question": row["query"], "page": page["text"]}


def query_state(row):
    return {"question": row["query"]}


def done_ids(path):
    return {json.loads(line)["query_id"] for line in open(path)} if os.path.exists(path) else set()

## 3. Kev-0.8B: their server (`kev.serve`) in their own uv environment, called on their `/v1/systemone` contract

In [ ]:
if RUN_KEV:
    subprocess.run(["pip", "install", "-q", "uv"], check=True)
    if not os.path.exists(KEV_DIR):
        subprocess.run(["git", "clone", "-q", KEV_REPO, KEV_DIR], check=True)
    subprocess.run(["git", "-C", KEV_DIR, "checkout", "-q", KEV_COMMIT], check=True)
    sync = subprocess.run(["uv", "sync", "--extra", "serve"], cwd=KEV_DIR, capture_output=True, text=True)
    open(f"{OUT}/logs/kev_sync.log", "w").write(sync.stdout + sync.stderr)
    if sync.returncode:
        print(sync.stderr[-3000:])
        raise SystemExit("uv sync failed")
    env = {**os.environ, "KEV_DTYPE": KEV_DTYPE}
    kev_log = open(f"{OUT}/logs/kev_serve.log", "w")
    kev_server = subprocess.Popen(["uv", "run", "--extra", "serve", "python", "-m", "kev.serve", "--run", KEV_MODEL,
                                   "--port", str(KEV_PORT)], cwd=KEV_DIR, env=env, stdout=kev_log, stderr=subprocess.STDOUT)
    import requests
    base = f"http://127.0.0.1:{KEV_PORT}"
    for _ in range(240):
        try:
            info = requests.get(f"{base}/v1/models", timeout=5).json()
            break
        except Exception:
            if kev_server.poll() is not None:
                print(open(f"{OUT}/logs/kev_serve.log").read()[-3000:])
                raise SystemExit("kev.serve exited")
            time.sleep(5)
    else:
        raise SystemExit("kev.serve did not come up in 20 minutes")
    print(json.dumps(info)[:1000])

In [ ]:
from concurrent.futures import ThreadPoolExecutor


def kev_noul(state, instructions):
    body = {"state": state, "model": "kev-latest", "questions": {"q": {"type": "noul", "instructions": instructions}}}
    r = requests.post(f"{base}/v1/systemone", json=body, timeout=600)
    if r.status_code != 200:
        return None, f"{r.status_code}: {r.text[:300]}"
    return r.json()["answers"]["q"]["noul"], None


if RUN_KEV:
    path = f"{OUT}/kev_scores.jsonl"
    skip = done_ids(path)
    print("smoke:", kev_noul(page_state(rows[0], rows[0]["pages"][0]), PAGE_QUESTION))
    started = time.time()
    with open(path, "a") as sink, ThreadPoolExecutor(KEV_WORKERS) as pool:
        for i, row in enumerate(rows):
            if row["query_id"] in skip:
                continue
            jobs = [pool.submit(kev_noul, page_state(row, p), PAGE_QUESTION) for p in row["pages"]]
            jobs.append(pool.submit(kev_noul, query_state(row), QUERY_QUESTION))
            got = [j.result() for j in jobs]
            sink.write(json.dumps({"query_id": row["query_id"], "model": KEV_MODEL, "dtype": KEV_DTYPE,
                                   "p_page": [g[0] for g in got[:-1]], "p_query": got[-1][0],
                                   "errors": [g[1] for g in got if g[1]]}) + "\n")
            sink.flush()
            if (i + 1) % 20 == 0:
                print(f"kev {i + 1}/{len(rows)} queries, {time.time() - started:.0f}s")
    kev_server.terminate()
    kev_server.wait()
    print("kev done")

## 4. Laya-multilingual: their `Router.predict_batch`, checkpoint pinned to multilingual

In [ ]:
if RUN_LAYA:
    subprocess.run(["pip", "install", "-q", LAYA_SPEC], check=True)

In [ ]:
if RUN_LAYA:
    import laya
    from laya import Router
    print("laya", laya.__version__)
    router = Router(device="cuda")
    page_q = {"q": {"type": "noul", "instructions": PAGE_QUESTION}}
    query_q = {"q": {"type": "noul", "instructions": QUERY_QUESTION}}
    smoke = router.predict(page_state(rows[0], rows[0]["pages"][0]), page_q, model=LAYA_MODEL)
    print("smoke:", smoke["answers"]["q"], smoke["routing"])
    path = f"{OUT}/laya_scores.jsonl"
    skip = done_ids(path)
    started = time.time()
    with open(path, "a") as sink:
        for i, row in enumerate(rows):
            if row["query_id"] in skip:
                continue
            reqs = [{"state": page_state(row, p), "questions": page_q, "model": LAYA_MODEL} for p in row["pages"]]
            reqs.append({"state": query_state(row), "questions": query_q, "model": LAYA_MODEL})
            out = router.predict_batch(reqs, batch_size=LAYA_BATCH)
            models = sorted({o["routing"]["model"] for o in out})
            sink.write(json.dumps({"query_id": row["query_id"], "model": LAYA_MODEL, "routed": models,
                                   "p_page": [o["answers"]["q"]["noul"] for o in out[:-1]],
                                   "p_query": out[-1]["answers"]["q"]["noul"]}) + "\n")
            sink.flush()
            if (i + 1) % 20 == 0:
                print(f"laya {i + 1}/{len(rows)} queries, {time.time() - started:.0f}s")
    print("laya done")

## 5. Summary and download

In [ ]:
for name in ("kev_scores.jsonl", "laya_scores.jsonl"):
    path = f"{OUT}/{name}"
    if not os.path.exists(path):
        print(name, "missing")
        continue
    got = [json.loads(line) for line in open(path)]
    probs = [p for g in got for p in g["p_page"] if p is not None]
    errors = sum(len(g.get("errors", [])) for g in got)
    print(f"{name}: {len(got)} queries, {len(probs)} page probs, errors {errors}, "
          f"mean p_page {sum(probs) / max(len(probs), 1):.3f}, share > 0.5 {sum(p > 0.5 for p in probs) / max(len(probs), 1):.3f}")
subprocess.run(["zip", "-qr", f"/kaggle/working/{ZIP}", "system1_out"], cwd="/kaggle/working", check=True)
print(f"download /kaggle/working/{ZIP}")